In [15]:
# 1. Install NeMo and dependencies (This takes 2-3 mins)
!pip install -q nemo_toolkit[asr] datasets torch

import torch
import nemo.collections.asr as nemo_asr
from datasets import load_dataset, Audio
from google.colab import userdata
import numpy as np

def run_nemo_conformer_vaani():
    # 2. Hardware Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 3. Load NVIDIA Conformer Model
    # This automatically downloads the .nemo weights from Hugging Face/NGC
    MODEL_ID = "nvidia/stt_en_conformer_ctc_large"
    print(f"Loading {MODEL_ID}...")
    model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(MODEL_ID).to(device)

    # 4. Configuration for VAANI
    LANGUAGE_SUBSET = "AndhraPradesh_Anantpur"
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None

    # 5. Load VAANI Dataset
    print(f"Streaming VAANI ({LANGUAGE_SUBSET})...")
    ds = load_dataset(
        "ARTPARK-IISc/Vaani",
        LANGUAGE_SUBSET,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    # NeMo Conformer models require 16kHz
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))

    # 6. Inference Loop
    print("\n" + "="*30)
    for i, sample in enumerate(ds.take(6)):
        audio_array = sample["audio"]["array"]
        ground_truth = sample.get("transcription", "No transcript found")

        # NeMo .transcribe() expects a filepath or a list of arrays
        # Since we are streaming, we pass the array directly
        # We need to save it to a temp buffer or use the tensors directly

        # Method: Convert to tensor and use the model's forward pass or transcribe
        # Note: NeMo's transcribe is highly optimized for files; for arrays we do:
        with torch.no_grad():
            # Create a dummy manifest/metadata for the audio
            # Or simpler: save to a temporary wav file
            import soundfile as sf
            temp_path = f"temp_sample_{i}.wav"
            sf.write(temp_path, audio_array, 16000)

            # Transcribe
            prediction = model.transcribe([temp_path])[0]

        print(f"[Sample {i+1}]")
        print(f"Expected: {ground_truth}")
        print(f"AI Said : {prediction}") # NeMo models usually output lowercase
        print("-" * 30)

if __name__ == "__main__":
    run_nemo_conformer_vaani()

Using device: cpu
Loading nvidia/stt_en_conformer_ctc_large...
[NeMo I 2026-04-07 10:43:48 mixins:184] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-04-07 10:43:48 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-04-07 10:43:48 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-04-07 10:43:50 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_en_conformer_ctc_large/snapshots/5f18b90411b9fdfc9b52b8caf636f204491afb01/stt_en_conformer_ctc_large.nemo.
Streaming VAANI (AndhraPradesh_Anantpur)...


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

...

[Sample 4]
Expected: No transcript found
AI Said : Hypothesis(score=tensor(-8.0760), y_sequence=tensor([128, 128,  87, 128, 128, 128, 128,  87,  20,  20, 128,  20, 128,  69,
        128, 128, 128, 128,   6,   7, 128,  35, 128, 128,  68, 128, 128, 128,
         11, 128,  19,  30,   3, 128,   3,   3,   4,   3, 128,   7,   4, 128,
        128]), text='e egg sodakit the sweeteat', dec_out=None, dec_state=None, timestamp=[], alignments=None, frame_confidence=None, token_confidence=None, word_confidence=None, length=0, y=None, lm_state=None, lm_scores=None, ngram_lm_state=None, tokens=None, last_token=None, token_duration=None, last_frame=None, biasing_cfg=None, xatt_scores=None)
------------------------------


Transcribing: 1it [00:00,  2.24it/s]
[NeMo W 2026-04-07 10:44:13 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-07 10:44:13 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[Sample 5]
Expected: No transcript found
AI Said : Hypothesis(score=tensor(-4.6085), y_sequence=tensor([128, 128, 113, 128, 128, 128, 128, 128,  67, 128, 128, 128, 128, 128,
        128, 128,  12, 128, 128, 128, 128, 128, 128,  51, 128, 128,   2, 116,
        128, 128, 128,  59, 128, 128,   2, 128, 128, 128, 128, 128, 128, 128,
        128, 128, 128]), text='uh n aro j c ', dec_out=None, dec_state=None, timestamp=[], alignments=None, frame_confidence=None, token_confidence=None, word_confidence=None, length=0, y=None, lm_state=None, lm_scores=None, ngram_lm_state=None, tokens=None, last_token=None, token_duration=None, last_frame=None, biasing_cfg=None, xatt_scores=None)
------------------------------


Transcribing: 1it [00:00,  1.06it/s]

[Sample 6]
Expected: No transcript found
AI Said : Hypothesis(score=tensor(-16.7849), y_sequence=tensor([128,  25, 128, 128, 128, 128,   2, 128, 128, 128,  64, 128, 128, 128,
         86, 128, 128, 128,  20, 128,   7, 128, 128, 128,   6, 128,   7, 128,
        128, 128, 128, 128, 128, 128, 128, 128,   2,  63, 128,  38, 128, 128,
         86, 128, 128, 128, 128, 128, 128,  25, 128, 128, 128, 128, 128,   2,
         63, 128,  38, 128, 128,  86, 128, 128, 128,   2, 128, 128, 128, 128,
        128, 128,  35, 128, 128, 128,   5, 128, 128, 128, 128, 128, 128, 128,
        128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128,
        128,  52, 128, 128, 128,  47, 128, 128, 128,   3, 128, 128, 128,  40,
        128,  21, 128,   5, 128,  68, 128, 128,  40,  21, 128,   5, 128,  68,
        128,  19, 128, 128, 128, 128,  67, 128,  10, 128,  67, 128,  10, 128,
         15, 128,   8, 128,   2,  63, 128,   3, 128, 128, 128,  19, 128,   5,
          9, 128, 128, 128,  38, 128, 128,   